<a href="https://colab.research.google.com/github/fabriciosantana/unb/blob/main/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projeto-02/QAFD_RAG_Discursos_Senado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QAFD-RAG — Pronunciamentos do Senado (56ª Legislatura)

Notebook exclusivo para consultar o dataset [fabriciosantana/discursos-senado-legislatura-56](https://huggingface.co/datasets/fabriciosantana/discursos-senado-legislatura-56). Inspirado no `QAFD_RAG.ipynb` deste projeto, com execução no **Google Colab, Conda e Google Drive**.

Reutiliza os componentes passage-entity do [QAFD-RAG](https://github.com/Tarzanagh/QAFD-RAG), sem alterar seus arquivos. O arquivo `senado_pipeline.py`, versionado junto deste notebook, prepara os discursos, constrói o grafo, recupera trechos e gera respostas em português com referências.

**Configuração inicial:** piloto de 20 discursos com texto integral; chunks de aproximadamente 400 tokens; embeddings locais `jina-v3`; extração e respostas com `gpt-4o-mini` via API. As células de construção e consulta consomem API. A GPU do Colab é opcional para Jina, mas acelera a execução; CPU pode ser lenta. O modelo de geração não roda no Colab.

**Persistência:** dados, logs, grafos e resultados ficam no Drive. O script de integração é lido da cópia do repositório. Textos integrais ausentes são excluídos; resumos não são tratados como discursos completos. As respostas se referem apenas ao corpus selecionado.

Execute as células em ordem. O notebook foi verificado localmente com dados sintéticos e sem chamadas pagas; a execução integral no Colab depende de suas credenciais e recursos.


## 1. Preparar Conda no Colab

Na primeira execução, `condacolab.install()` pode reiniciar o kernel. Aguarde a reconexão e execute novamente esta célula; depois continue. Nenhum `conda activate` é necessário. As dependências e o pipeline serão executados com `conda run` no ambiente `qafd-rag`.


In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

✨🍰✨ Everything looks OK!


In [ ]:
!conda --version

conda 26.3.2


## 2. Montar o Drive e definir o piloto

Os filtros de autor, data e tema são aplicados **na seleção do corpus antes da construção**. Eles não são filtros por pergunta sobre um grafo maior. O filtro de tema faz busca literal em texto, resumo e indexação; não é uma classificação temática.

Para aumentar o piloto, mude `MAX_SPEECHES`. `None` seleciona todos os discursos elegíveis e pode gerar muitos chunks, custos de API e processamento prolongado. Alterações no corpus, configuração ou commit criam outra pasta de execução.

Mantenha `senado_pipeline.py` e este notebook na sua cópia do projeto no Drive. Ajuste `SENADO_PROJECT_DIR` abaixo para a pasta que contém o script. Abrir apenas o notebook pelo GitHub não disponibiliza automaticamente os outros arquivos no Colab.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import json
import os
import subprocess

QAFD_RAG_DIR = Path("/content/drive/MyDrive/Colab Notebooks/QAFD-RAG")
# Ajuste para a pasta da sua cópia do repositório unb no Drive.
SENADO_PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/unb/01-ppgi0034-redes-neurais-e-aprendizado-profundo/projeto-02")
WORK_ROOT = QAFD_RAG_DIR / "notebooks/senado"
CONDA_ENV = "qafd-rag"
REPO_REF = "main"  # Usado apenas ao clonar. O commit efetivo será registrado.

MAX_SPEECHES = 20
AUTHOR = None           # Exemplo: "Paulo Paim"; busca literal parcial.
TOPIC = None            # Exemplo: "previdência"; não ignora acentos.
DATE_START = None       # Exemplo: "2019-01-01"
DATE_END = None         # Exemplo: "2019-12-31"
CHUNK_TOKENS = 400      # Tokenizador cl100k_base; não é o tokenizador do Jina.
OVERLAP_WORDS = 40      # Sobreposição em palavras; chunks respeitam fronteiras de palavras.
EMBEDDING_BATCH_SIZE = 4
LLM_CONCURRENCY = 2
MAX_ITERATIONS = 5000
RETRIEVAL_TOP_K = 10

assert MAX_SPEECHES is None or (isinstance(MAX_SPEECHES, int) and MAX_SPEECHES > 0)
assert CHUNK_TOKENS > 0 and OVERLAP_WORDS >= 0
assert EMBEDDING_BATCH_SIZE > 0 and LLM_CONCURRENCY > 0


## 3. Obter o código e instalar dependências

O clone existente é preservado, incluindo eventuais alterações suas. O notebook registra seu commit; para reprodução estrita, use uma cópia limpa com commit fixado. Não crie outro clone sobre a pasta atual.


In [ ]:
!git clone https://github.com/Tarzanagh/QAFD-RAG.git /content/drive/MyDrive/Colab\ Notebooks/QAFD-RAG

In [ ]:
%ls "{QAFD_RAG_DIR}"

benchmarks/             data/  kg/       README.md         run.sh
condacolab_install.log  docs/  outputs/  requirements.txt  src/


In [ ]:
%cd "{QAFD_RAG_DIR}"

/content/drive/MyDrive/Colab Notebooks/QAFD-RAG


In [ ]:
%ls

benchmarks/             data/  kg/       README.md         run.sh
condacolab_install.log  docs/  outputs/  requirements.txt  src/


## 4. Credencial de API

Crie o segredo `OPENAI_API_KEY` no painel de chaves do Colab e habilite seu acesso ao notebook. A chave é enviada ao subprocesso pelo ambiente e não é gravada nos artefatos.


In [ ]:
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY").strip()
assert os.environ["OPENAI_API_KEY"], "Chave vazia."
print("Chave configurada; valor oculto.")


Chave configurada; valor oculto.


## 5. Localizar a integração do repositório

A próxima célula localiza `senado_pipeline.py` em `SENADO_PROJECT_DIR`. Ela não gera scripts nem modifica o código do QAFD-RAG.

A extração e a recuperação reutilizam os componentes originais, incluindo seus prompts. A resposta final usa instruções em português, citações `[F1]`, `[F2]` e orientação para distinguir autor registrado de falas de terceiros. Os metadados são preservados fora dos vetores e incorporados aos trechos para contextualização; isso não garante atribuição perfeita. Valide amostras da extração antes de ampliar o corpus.

Falhas de API durante a construção impedem que ela seja marcada como concluída. Os logs ainda precisam ser conferidos para problemas de parsing ou avisos de não convergência. O notebook limita concorrência e registra tokens retornados pela API.


In [ ]:
INTEGRATION = (SENADO_PROJECT_DIR / "senado_pipeline.py").resolve()
if not INTEGRATION.is_file():
    raise FileNotFoundError(
        f"Integração não encontrada: {INTEGRATION}. "
        "Coloque senado_pipeline.py do repositório nessa pasta do Drive "
        "ou ajuste SENADO_PROJECT_DIR."
    )
print("Integração do repositório:", INTEGRATION)
conda_run(["python", INTEGRATION, "--help"])


## 6. Baixar, selecionar e dividir os discursos

Usa o Parquet da versão `v1.1.1`. Cada trecho mantém identificador do discurso, autor, data, partido registrado, UF e URL. Cabeçalhos administrativos são preservados para evitar remover falas por heurística; confira abaixo o primeiro trecho.

A preparação é local: não chama a API de geração. Um identificador calculado a partir da configuração e dos textos separa os caches de cada corpus.


In [ ]:
import hashlib

SETTINGS = {
    "dataset_id": "fabriciosantana/discursos-senado-legislatura-56",
    "dataset_filename": "data/full/discursos_2019-02-01_2023-01-31.parquet",
    "dataset_revision": "v1.1.1",
    "qafd_commit": COMMIT,
    "tracked_changes": CODE_STATUS,
    "tracked_diff_sha256": hashlib.sha256(subprocess.check_output(["git", "diff", "HEAD"], cwd=QAFD_RAG_DIR)).hexdigest(),
    "integration_sha256": hashlib.sha256(INTEGRATION.read_bytes()).hexdigest(),
    "max_speeches": MAX_SPEECHES, "seed": 42,
    "author": AUTHOR, "topic": TOPIC, "date_start": DATE_START, "date_end": DATE_END,
    "chunk_tokens": CHUNK_TOKENS, "overlap_words": OVERLAP_WORDS,
    "embedding": "jina-v3", "embedding_batch_size": EMBEDDING_BATCH_SIZE,
    "llm": "gpt-4o-mini", "llm_concurrency": LLM_CONCURRENCY,
    "max_iterations": MAX_ITERATIONS, "retrieval_top_k": RETRIEVAL_TOP_K,
    "runs_root": str(WORK_ROOT / "runs"),
}
SETTINGS_PATH = WORK_ROOT / "settings.json"
ACTIVE_PATH = WORK_ROOT / "active_run.json"
SETTINGS_PATH.write_text(json.dumps(SETTINGS, ensure_ascii=False, indent=2), encoding="utf-8")
conda_run(["python", INTEGRATION, "prepare", "--settings", SETTINGS_PATH, "--active", ACTIVE_PATH])
RUN_DIR = Path(json.loads(ACTIVE_PATH.read_text())["run_dir"])
CHUNKS = json.loads((RUN_DIR / "chunks.json").read_text())
print("Pasta da execução:", RUN_DIR)
print("Primeiro chunk:\n", CHUNKS[0]["content"])


## 7. Construir ou reutilizar o grafo — consome API

Execute após conferir o corpus. O texto integral pode conter várias falas no mesmo pronunciamento. O custo depende do **número de chunks**, não apenas do número de discursos; a estimativa feita para MuSiQue não se transfere diretamente para esta base.

O grafo concluído é reutilizado se você repetir a mesma preparação. Não há reconstrução forçada por padrão. Se houver falhas de LLM, confira o log e a conta da API antes de repetir; a repetição de uma construção incompleta pode refazer chamadas.


In [ ]:
from datetime import datetime, timezone

LOG_DIR = RUN_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)

def execute_pipeline(command, *extra):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    log_path = LOG_DIR / f"{command}_{stamp}.log"
    arguments = ["conda", "run", "--no-capture-output", "-n", CONDA_ENV,
                 "python", "-u", str(INTEGRATION), command,
                 "--project", str(QAFD_RAG_DIR), "--run-dir", str(RUN_DIR), *map(str, extra)]
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(arguments, cwd=QAFD_RAG_DIR,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1, start_new_session=True)
        try:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            returncode = process.wait()
        finally:
            if process.poll() is None:
                # Cancela também o Python filho de conda run.
                import signal
                os.killpg(process.pid, signal.SIGTERM)
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait()
    if returncode:
        raise RuntimeError(f"Falha no comando {command}; confira {log_path}")
    print("Log:", log_path)

execute_pipeline("build")
BUILD = json.loads((RUN_DIR / "build_complete.json").read_text())
GRAPH_DIR = Path(BUILD["graph_dir"])
print(BUILD)


## 8. Fazer uma pergunta — consome API

Substitua a pergunta e execute esta célula para cada consulta. O contexto contém os cinco primeiros trechos recuperados. Os scores são pontuações da recuperação, **não probabilidades de verdade**.

Não há respostas de referência nesta base; este notebook não calcula Exact Match, F1 ou Recall automaticamente. Compare manualmente a resposta e as evidências. Uma resposta com citações pode ainda conter erros de interpretação. Para perguntas sobre todo o Senado, lembre que o piloto contém apenas os discursos selecionados.


In [ ]:
QUESTION = "Quais temas e argumentos aparecem nos pronunciamentos recuperados? Indique os autores registrados e cite as fontes."

question_file = RUN_DIR / "question_input.json"
question_file.write_text(json.dumps({"question": QUESTION}, ensure_ascii=False), encoding="utf-8")
execute_pipeline("query", "--question-file", question_file)


## 9. Inspecionar a resposta e os trechos utilizados

O resultado completo fica em `queries/`; `latest_query.json` aponta para o conteúdo da consulta mais recente. Não é um relatório de benchmark MuSiQue.


In [ ]:
from IPython.display import Markdown, display

result = json.loads((RUN_DIR / "latest_query.json").read_text())
display(Markdown(result["answer"]))
if result["citation_warning"]:
    print(result["citation_warning"])
for evidence in result["evidence"]:
    print("\n" + "=" * 70)
    print(f"[{evidence['citation']}] {evidence['author']} — {evidence['date']}")
    print("Pronunciamento:", evidence["speech_id"], "| Chunk:", evidence["chunk_id"])
    print("Partido / UF:", evidence["party"], evidence["state"])
    print("Fonte:", evidence["url"])
    print("Score:", evidence["score"])
    print(evidence["text"])


## 10. Exportar grafo e embeddings — sem chamadas à API

Gera `graph.graphml` ao lado de `graph.pickle`, para [Gephi Lite](https://lite.gephi.org/). Os nós de passagem recebem atributos de origem; entidades compartilhadas não recebem um único autor.

Em cada pasta de embeddings, gera `vectors.tsv`, `metadata.tsv` (conteúdo + ID) e `labels.tsv` (somente conteúdo, sem cabeçalho). No [Embedding Projector](https://projector.tensorflow.org/), carregue `vectors.tsv` e `labels.tsv` da mesma coleção para evitar rótulos com hashes.


In [ ]:
execute_pipeline("export")
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file() and path.suffix in {".graphml", ".parquet", ".tsv"}:
        print(path.relative_to(RUN_DIR))


In [ ]:
# Opcional: baixar o grafo para abrir no Gephi.
from google.colab import files
files.download(str(GRAPH_DIR / "graph.graphml"))


## 11. Conferir consumo de tokens

`api_usage.jsonl` registra o uso devolvido pela API nas chamadas concluídas. Não inclui chamadas que falharam antes de retornar uso, outros notebooks, nem custos de infraestrutura. Consulte o painel do provedor para o faturamento efetivo.


In [ ]:
usage_path = RUN_DIR / "api_usage.jsonl"
if usage_path.exists():
    records = [json.loads(line) for line in usage_path.read_text().splitlines() if line.strip()]
    prompt_tokens = sum((r.get("usage") or {}).get("prompt_tokens", 0) for r in records)
    completion_tokens = sum((r.get("usage") or {}).get("completion_tokens", 0) for r in records)
    print("Chamadas com uso registrado:", len(records))
    print("Tokens de entrada:", prompt_tokens)
    print("Tokens de saída:", completion_tokens)
else:
    print("Ainda não há consumo registrado nesta execução.")


## Arquivos e próximos experimentos

```text
QAFD-RAG/notebooks/senado/
├── settings.json
├── active_run.json
└── runs/senado_<identificador>/
    ├── config.json
    ├── corpus_stats.json
    ├── chunks.json
    ├── chunks_metadata.parquet
    ├── build_complete.json
    ├── api_usage.jsonl
    ├── logs/
    ├── queries/
    ├── latest_query.json
    └── artifacts/
        ├── openie_results_ner_gpt-4o-mini.json
        └── gpt-4o-mini_jina-v3/
            ├── graph.pickle
            ├── graph.graphml
            ├── chunk_embeddings/
            ├── entity_embeddings/
            └── fact_embeddings/
```

`senado_pipeline.py` fica em `projeto-02/`, junto do notebook, fora da árvore de saídas.

Para ampliar: mantenha o mesmo código, aumente `MAX_SPEECHES` ou ajuste os filtros e execute novamente desde a preparação. A nova configuração terá seu próprio grafo. Não atribua falta de recuperação à ausência de um tema em toda a base. Contagens exatas por autor, partido ou data devem ser feitas diretamente no Parquet.

Para uma avaliação sistemática, crie perguntas com respostas e trechos de suporte revisados manualmente. Compare também busca vetorial simples com QAFD no mesmo corpus. Mudanças de lote, modelo ou parâmetros podem alterar o custo e a recuperação.
